## Laboratorio 07 - Problema de CartPole con Deep Q-Learning 

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

### Creación de Entorno CartPole

In [2]:
env = gym.make('CartPole-v1')

print("¡Entorno CartPole creado exitosamente!\n")
print(f"Espacio de observación: {env.observation_space}")
print(f"Espacio de acción: {env.action_space}")
print(f"Número de acciones posibles: {env.action_space.n}")

state, info = env.reset()
print(f"Dimensiones del estado: {env.observation_space.shape}")
print(f"Estado inicial de ejemplo: {state}")

¡Entorno CartPole creado exitosamente!

Espacio de observación: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Espacio de acción: Discrete(2)
Número de acciones posibles: 2
Dimensiones del estado: (4,)
Estado inicial de ejemplo: [ 0.03249991  0.0322019   0.01386275 -0.03719383]


### Definición de las redes en línea y de destino

In [3]:
class DQN(nn.Module):
    """Red neuronal para Deep Q-Learning"""
    def __init__(self, state_size=4, action_size=2):
        super(DQN, self).__init__()
        
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64) 
        self.fc3 = nn.Linear(64, action_size)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [4]:
# Creación de 2 redes neuronales
online_net = DQN(state_size=4, action_size=2)  # Red en línea para selección de acciones
target_net = DQN(state_size=4, action_size=2)  # Red de destino para estimación de valores Q

In [5]:
# Red destino con mismos pesos que la red en línea (Solo inicialmente)
target_net.load_state_dict(online_net.state_dict())

<All keys matched successfully>

In [6]:
print("Redes neuronales creadas exitosamente!")
print(f"Red en línea: {online_net}")
print(f"Red de destino: {target_net}")
print("La red de destino ha sido inicializada con los mismos pesos que la red en línea.")

Redes neuronales creadas exitosamente!
Red en línea: DQN(
  (fc1): Linear(in_features=4, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=2, bias=True)
)
Red de destino: DQN(
  (fc1): Linear(in_features=4, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=2, bias=True)
)
La red de destino ha sido inicializada con los mismos pesos que la red en línea.


### Establecer Hiperparámetros

In [7]:
# Para el entrenamiento
NUM_EPISODES = 1000          
BATCH_SIZE = 32              

# Factor de descuento
GAMMA = 0.99                 

# Parámetros de exploración (epsilon-greedy)
EPSILON = 1.0                # 100%
EPSILON_DECAY = 0.995      
EPSILON_MIN = 0.01           # 1% exploración mínima

# Adicionales para optimización
LEARNING_RATE = 0.001        # Tasa de aprendizaje
MEMORY_SIZE = 10000          # Tamaño del buffer de experiencia
TARGET_UPDATE_FREQ = 10      # Frecuencia de actualización de la red de destino (cada N episodios)

### Selección de acciones ε-greedy

In [8]:
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
online_net.to(device)
target_net.to(device)

def select_action_epsilon_greedy(state: np.ndarray, epsilon: float) -> int:
    """
    Política ε-greedy sobre la red en línea.
    - Con prob. ε: explora (acción aleatoria)
    - Con prob. 1-ε: explota (argmax Q(s,·))
    """
    if random.random() < epsilon:
        return env.action_space.sample()
    with torch.no_grad():
        s = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        q_values = online_net(s)
        return int(torch.argmax(q_values, dim=1).item())


### Reproducción de la experiencia (Experience Replay)

In [9]:
from collections import deque, namedtuple

Transition = namedtuple("Transition", ("state", "action", "reward", "next_state", "done"))

class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)
    def __len__(self):
        return len(self.buffer)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append(Transition(state, action, reward, next_state, done))
    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        # Empaquetar en tensores en el dispositivo
        states      = torch.as_tensor(np.array([b.state for b in batch], dtype=np.float32), device=device)
        actions     = torch.as_tensor([b.action for b in batch], dtype=torch.long,  device=device).unsqueeze(1)
        rewards     = torch.as_tensor([b.reward for b in batch], dtype=torch.float32, device=device).unsqueeze(1)
        next_states = torch.as_tensor(np.array([b.next_state for b in batch], dtype=np.float32), device=device)
        dones       = torch.as_tensor([b.done for b in batch], dtype=torch.float32, device=device).unsqueeze(1)
        return states, actions, rewards, next_states, dones

memory = ReplayBuffer(MEMORY_SIZE)


### Ciclo de entrenamiento (con actualización periódica de la red de destino)

In [10]:
optimizer = torch.optim.Adam(online_net.parameters(), lr=LEARNING_RATE)
loss_fn   = nn.SmoothL1Loss()  

def optimize_from_replay():
    """
    Toma un minibatch del buffer y hace un paso de optimización de DQN.
    """
    if len(memory) < BATCH_SIZE:
        return None 

    states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

    q_sa = online_net(states).gather(1, actions)

    with torch.no_grad():
        max_next_q = target_net(next_states).max(dim=1, keepdim=True)[0]
        targets = rewards + GAMMA * (1.0 - dones) * max_next_q

    loss = loss_fn(q_sa, targets)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(online_net.parameters(), 1.0) 
    optimizer.step()

    return float(loss.item())

epsilon = EPSILON
episode_rewards = []
episode_losses  = []

for ep in range(1, NUM_EPISODES + 1):
    state, _ = env.reset()
    ep_reward = 0.0
    ep_losses = []

    done = False
    while not done:
        # 1) Selección de acción ε-greedy
        action = select_action_epsilon_greedy(state, epsilon)

        # 2) Interacción con el entorno
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # 3) Guardar transición en el buffer
        memory.push(state, action, reward, next_state, float(done))

        # 4) Actualizar red en línea a partir del replay buffer
        loss_val = optimize_from_replay()
        if loss_val is not None:
            ep_losses.append(loss_val)

        # 5) Avanzar estado/acumular recompensa
        state = next_state
        ep_reward += reward

    epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)

    if ep % TARGET_UPDATE_FREQ == 0:
        target_net.load_state_dict(online_net.state_dict())

    episode_rewards.append(ep_reward)
    episode_losses.append(np.mean(ep_losses) if ep_losses else np.nan)

    if ep % 10 == 0:
        avg10 = np.mean(episode_rewards[-10:])
        print(f"Ep {ep:04d} | Recompensa: {ep_reward:6.1f} | Prom(últ.10): {avg10:6.1f} | "
              f"ε={epsilon:0.3f} | Memoria={len(memory)} | Loss={np.nanmean(episode_losses[-10:]):.4f}")


Ep 0010 | Recompensa:   20.0 | Prom(últ.10):   19.6 | ε=0.951 | Memoria=196 | Loss=0.0619
Ep 0020 | Recompensa:   22.0 | Prom(últ.10):   20.5 | ε=0.905 | Memoria=401 | Loss=0.0415
Ep 0030 | Recompensa:   13.0 | Prom(últ.10):   29.4 | ε=0.860 | Memoria=695 | Loss=0.0660
Ep 0040 | Recompensa:   25.0 | Prom(últ.10):   32.4 | ε=0.818 | Memoria=1019 | Loss=0.0773
Ep 0050 | Recompensa:   33.0 | Prom(últ.10):   34.9 | ε=0.778 | Memoria=1368 | Loss=0.0840
Ep 0060 | Recompensa:   69.0 | Prom(últ.10):   35.0 | ε=0.740 | Memoria=1718 | Loss=0.0943
Ep 0070 | Recompensa:   23.0 | Prom(últ.10):   34.9 | ε=0.704 | Memoria=2067 | Loss=0.1095
Ep 0080 | Recompensa:   17.0 | Prom(últ.10):   40.4 | ε=0.670 | Memoria=2471 | Loss=0.1210
Ep 0090 | Recompensa:   20.0 | Prom(últ.10):   30.7 | ε=0.637 | Memoria=2778 | Loss=0.1473
Ep 0100 | Recompensa:   19.0 | Prom(últ.10):   45.2 | ε=0.606 | Memoria=3230 | Loss=0.1422
Ep 0110 | Recompensa:  129.0 | Prom(últ.10):   65.2 | ε=0.576 | Memoria=3882 | Loss=0.1698
Ep